In [1]:
import pandas as pd
flows = pd.read_excel('1_F_Wood_Waste_Germany_2007_2010_2015_UBA.xlsx','Data') # Insert filename of IEDC download
flows

,commodity,material,layer,region,time,origin_process,destination_process,scenario,value,unit nominator,unit denominator,stats_array string,comment
0,wood waste,"wood, wood waste, and wood products",dry mass,Germany,2007,forestry,waste wood collection,History,212,kt,yr,none,none
1,wood waste,"wood, wood waste, and wood products",dry mass,Germany,2007,wood-processing industry,waste wood collection,History,2831,kt,yr,none,none
2,"waste, packaging","wood, wood waste, and wood products",dry mass,Germany,2007,waste management,waste wood collection,History,674,kt,yr,none,none
3,construction and demolition waste,"wood, wood waste, and wood products",dry mass,Germany,2007,waste management,waste wood collection,History,3888,kt,yr,none,none
4,"wood waste, in MSW, separately recorded","wood, wood waste, and wood products",dry mass,Germany,2007,waste management,waste wood collection,History,718,kt,yr,none,none
...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,"wood, recycled","wood, wood waste, and wood products",dry mass,Germany,2015,recycling,production of particle board,History,1100,kt,yr,none,none
77,roundwood,"wood, wood waste, and wood products",dry mass,Germany,2015,sawmill,production of particle board,History,4126,kt,yr,none,none
78,particle board,"wood, wood waste, and wood products",dry mass,Germany,2015,production of particle board,export of particle board,History,1003,kt,yr,none,none
79,particle board,"wood, wood waste, and wood products",dry mass,Germany,2015,import of particle board,domestic market for particle boards,History,1441,kt,yr,none,none


In [2]:
import pandas as pd
flows       = pd.read_excel('1_F_Wood_Waste_Germany_2007_2010_2015_UBA.xlsx','Data')
material    = 'wood, wood waste, and wood products'
layer       = 'dry mass'
region      = 'Germany'
scenario    = 'History'
time        =  2015 # 2007, 2010, 2015

ExtractData = flows.loc[(flows['aspect 2 : material'] == material) & (flows['aspect 3 : layer'] == layer) & (flows['aspect 4 : region'] == region) & (flows['aspect 8 : scenario'] == scenario) & (flows['aspect 5 : time'] == time)]
SankeyDatar = ExtractData.drop(columns=['id','dataset_name','aspect 2 : material','aspect 3 : layer','aspect 4 : region','aspect 8 : scenario','aspect 5 : time','aspect 9 : ','aspect 10 : ','aspect 11 : ','aspect 12 : ','unitcode','unitcode1','stats_array_1','stats_array_2','stats_array_3','stats_array_4','comment','reserve1','reserve2','reserve3'])
SankeyDatar.rename(columns={'aspect 6 : origin_process': 'source', 'aspect 7 : destination_process': 'target','aspect 1 : commodity': 'type'}, inplace=True)
column_names= ['source','target','type','value']
SankeyDatar = SankeyDatar.reindex(columns=column_names)
SankeyDatar.reset_index(drop=True, inplace=True)
SankeyDatar

,source,target,type,value
0,forestry,waste wood collection,wood waste,711
1,wood-processing industry,waste wood collection,wood waste,2755
2,waste management,waste wood collection,"waste, packaging",895
3,waste management,waste wood collection,construction and demolition waste,4491
4,waste management,waste wood collection,"wood waste, in MSW, separately recorded",719
5,waste management,waste wood collection,bulky waste,998
6,waste management,waste wood collection,"wood waste, in MSW",605
7,import of waste,waste wood processing,wood waste,785
8,waste wood collection,export of waste,wood waste,12
9,waste wood collection,waste wood processing,wood waste,11162


In [3]:
from floweaver import *

Sankey_version = '1_F_Wood_Waste_Germany_2007_2010_2015_UBA'

# Define the nodes
nodes = {
    'forestry': ProcessGroup(['forestry'], title='forestry'),
    'wood-processing industry': ProcessGroup(['wood-processing industry'], title='wood processing'),
    'waste management': ProcessGroup(['waste management'], title='waste management'),
    'waste wood collection': ProcessGroup(['waste wood collection'], title='collection'),
    'import of waste': ProcessGroup(['import of waste'], title='imports'),
    'waste wood processing': ProcessGroup(['waste wood processing'], title='processing'), 
    'export of waste': ProcessGroup(['waste wood export'], title='exports'),
    'mechanical-biological treatment': ProcessGroup(['mechanical-biological treatment'], title='MBA'),
    'shredding, sorting, and construction waste treatment': ProcessGroup(['shredding, sorting, and construction waste treatment'], title='waste treatment'),
    'export of waste': ProcessGroup(['export of waste'], title='export'),
    'large combustion plants and co-firing plants': ProcessGroup(['large combustion plants and co-firing plants'], title='large combustion plants and co-firing plants'),
    'small and medium-sized combustion plants': ProcessGroup(['small and medium-sized combustion plants'], title='small and medium-sized combustion plants'),
    'recycling': ProcessGroup(['recycling'], title='recycling'),
    'waste incineration plant': ProcessGroup(['waste incineration plant'], title='waste incineration plant'),
    'way1': Waypoint(title='', direction='R'),
    'way2': Waypoint(title='', direction='R'),
    'way3': Waypoint(title='', direction='R'),
    'production of particle board': ProcessGroup(['production of particle board'], title='production of particle board'),
    'export of recycled wood': ProcessGroup(['export of recycled wood'], title='exports'),
    'sawmill': ProcessGroup(['sawmill'], title='sawmill'), 
    'import of particle board': ProcessGroup(['import of particle board'], title='import'),
    'domestic market for particle boards': ProcessGroup(['domestic market for particle boards'], title='domestic market'),
    'export of particle board': ProcessGroup(['export of particle board'], title='exports'),
}

# Define the bundles
bundles = [
    Bundle('forestry', 'waste wood collection'),
    Bundle('wood-processing industry', 'waste wood collection'),
    Bundle('waste management', 'waste wood collection'),
    Bundle('import of waste', 'waste wood processing'), 
    Bundle('waste wood collection', 'waste wood processing'),      
    Bundle('waste wood collection', 'export of waste', waypoints=['way3','way2']),  
    Bundle('waste wood processing', 'waste incineration plant', waypoints=['way1']),  
    Bundle('waste wood processing', 'shredding, sorting, and construction waste treatment'), 
    Bundle('waste wood processing', 'large combustion plants and co-firing plants'), 
    Bundle('waste wood processing', 'mechanical-biological treatment'), 
    Bundle('mechanical-biological treatment', 'waste incineration plant'), 
    Bundle('mechanical-biological treatment', 'large combustion plants and co-firing plants'), 
    Bundle('shredding, sorting, and construction waste treatment', 'export of waste'), 
    Bundle('shredding, sorting, and construction waste treatment', 'waste incineration plant'), 
    Bundle('shredding, sorting, and construction waste treatment', 'large combustion plants and co-firing plants'), 
    Bundle('shredding, sorting, and construction waste treatment', 'recycling'),
    Bundle('shredding, sorting, and construction waste treatment', 'small and medium-sized combustion plants'),
    Bundle('recycling', 'production of particle board'),
    Bundle('recycling', 'export of recycled wood'),
    Bundle('sawmill', 'production of particle board'),
    Bundle('production of particle board', 'domestic market for particle boards'),
    Bundle('import of particle board', 'domestic market for particle boards'),
    Bundle('production of particle board','export of particle board'),
]

# Define the ordering
ordering = [
[['forestry'],['waste management'],['wood-processing industry']],
[['import of waste'],['waste wood collection']],
[[],['waste wood processing'],[],['way3']],
[[],['shredding, sorting, and construction waste treatment'],['mechanical-biological treatment'],['way1'],['way2']],
[['sawmill'],['large combustion plants and co-firing plants'],['recycling'],['small and medium-sized combustion plants'],['waste incineration plant'],['export of waste']],
[['import of particle board'],['production of particle board'],['export of recycled wood']],
[[],['domestic market for particle boards'],['export of particle board']],  
]

flows_by_type = Partition.Simple('type', [
    'wood waste','waste, packaging','construction and demolition waste',
    'wood waste, in MSW, separately recorded', 'bulky waste', 'wood waste, in MSW', 'waste wood, processed',
    'solid recovered fuel', 'wood, recycled', 'roundwood', 'particle board',
])

palette = {
    'wood waste': 'khaki',
    'waste, packaging': 'beige',
    'construction and demolition waste': 'wheat',
    'wood waste, in MSW, separately recorded': 'powderblue',
    'bulky waste': 'plum',
    'wood waste, in MSW': 'pink',
    'waste wood, processed': 'darkkhaki',
    'solid recovered fuel': 'lightgrey',
    'wood, recycled': 'olive',
    'roundwood': 'olivedrab',
    'particle board': 'darkseagreen',
}

# Generate the Sankey diagram
size = dict(width=1300, height=350)

# New SDD with the flow_partition set
sdd = SankeyDefinition(nodes, bundles, ordering, flow_partition=flows_by_type)

Sankey = weave(sdd, SankeyDatar, palette=palette).to_widget(**size)

Sankey # Show below

Sankey.auto_save_png(Sankey_version + '_' + region + '_' + str(time) + '.png')
Sankey.auto_save_svg(Sankey_version + '_' + region + '_' + str(time) + '.svg')

In [4]:
# double-check and export Sankey specification as .json
if isinstance(SankeyDatar, pd.DataFrame):
    SankeyDatar = Dataset(SankeyDatar)
    
spec = compile_sankey_definition(
    sdd,       # see above: sdd = SankeyDefinition(nodes, bundles, ordering, flow_partition=flows_by_type)
    measures   = "value",
    link_width = None,
    link_color = None,
    palette    = palette,
    add_elsewhere_waypoints = True,
    dimension_tables        = None,
)

Sankey = execute_weave(spec, SankeyDatar)
Sankey.to_widget(**size)

In [5]:
# export Sankey spec as .json for use in IEDC web application
import json
json_data = json.dumps(spec.to_json(), indent=2)
# Write Sankey spec to .json, filename as indicated in: 
# https://github.com/IndEcol/IEDC_floweaver_integration/blob/main/floweaver_IEDC_Sankey_configs/floweaver_IEDC_Sankey_configs.config
with open(Sankey_version + '.json', "w", encoding="utf-8") as f:
    f.write(json_data)
    f.write("\n")